<a href="https://colab.research.google.com/github/VaggelisApostolou/Challenges-in-Detecting-Toxic-Language-in-Greek-Sports-Social-Media/blob/main/Greek_BERT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Import Libraries**

In [ ]:
import os, random, numpy as np, pandas as pd, torch
import math
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim import AdamW
import torch.nn as nn
import torch.nn.functional as F
from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModelForSequenceClassification,
    AutoModelForMaskedLM,
    get_linear_schedule_with_warmup,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments
)
from datasets import load_dataset
import glob
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_recall_curve, classification_report, precision_score, recall_score, roc_auc_score
from sklearn.utils.class_weight import compute_class_weight

def set_seed(seed: int = 42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


device(type='cuda')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
MODEL_CHECKPOINT = "nlpaueb/bert-base-greek-uncased-v1"
TRAIN_FILE = "/content/drive/MyDrive/Toxic language in football Research/data/clean_corpus_reddit.json"
OUTPUT_DIR = "/content/drive/MyDrive/Toxic language in football Research/models/Greek-BERT"

# **Masked Language Modeling**

In [ ]:
BATCH_SIZE = 8
EPOCHS = 5 #Κανονικά 5
BLOCK_SIZE = 128 #Κανονικά 128

print("--- 1. Φόρτωση Δεδομένων ---")
datasets = load_dataset("json", data_files={"train": TRAIN_FILE})
print(f"Βρέθηκαν {len(datasets['train'])} παραδείγματα εκπαίδευσης.")

print("--- 2. Tokenization ---")
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=BLOCK_SIZE
    )

tokenized_datasets = datasets.map(
    tokenize_function,
    batched=True,
    num_proc=4,
    remove_columns=["text"]
)

print("--- 3. Προετοιμασία Μοντέλου ---")
model = AutoModelForMaskedLM.from_pretrained(MODEL_CHECKPOINT)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm_probability=0.15
)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="no",
    learning_rate=2e-5,
    weight_decay=0.01,
    per_device_train_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    save_strategy="epoch",
    logging_steps=50,
    fp16=True,
    push_to_hub=False,
)

print("--- 4. Έναρξη Εκπαίδευσης---")
small_train_dataset = tokenized_datasets["train"].select(range(2000))
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train_dataset, #tokenized_datasets["train"],
    data_collator=data_collator,
)

trainer.train()

print(f"--- 5. Αποθήκευση Μοντέλου στο {OUTPUT_DIR} ---")
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

#eval_results = trainer.evaluate()
#print(f"Perplexity: {math.exp(eval_results['eval_loss']):.2f}")

--- 1. Φόρτωση Δεδομένων ---
Βρέθηκαν 55606 παραδείγματα εκπαίδευσης.
--- 2. Tokenization ---


Map (num_proc=4):   0%|          | 0/55606 [00:00<?, ? examples/s]

--- 3. Προετοιμασία Μοντέλου ---


pytorch_model.bin:   0%|          | 0.00/454M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/204 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

--- 5. Αποθήκευση Μοντέλου στο /content/drive/MyDrive/Toxic language in football Research/models/Greek-BERT ---


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/Toxic language in football Research/models/Greek-BERT/tokenizer_config.json',
 '/content/drive/MyDrive/Toxic language in football Research/models/Greek-BERT/tokenizer.json')

# **Configuration**

In [ ]:
DATA_PATH = "/content/drive/MyDrive/Toxic language in football Research/final_labeled_batches/final_labeled.csv"
SEED      = 42
EPOCHS    = 8
BATCH_SZ  = 32
LR        = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10
MAX_LEN   = 256
PATIENCE  = 3
GRAD_CLIP = 1.0

MODEL_NAME = OUTPUT_DIR

LOSS_TYPE = "focal"   # "ce" or "focal"
FOCAL_GAMMA = 2.0

# Toxic class emphasis
TOXIC_WEIGHT_MULT = 1.5
USE_SAMPLER = True

# Threshold objective
FBETA = 2.0              # tune threshold on F_beta (beta=2 -> recall emphasis)

set_seed(SEED)
print("Device:", device)
print("Loss:", LOSS_TYPE, "| gamma:", FOCAL_GAMMA, "| toxic_w_mult:", TOXIC_WEIGHT_MULT, "| Fbeta:", FBETA)


Device: cuda
Loss: focal | gamma: 2.0 | toxic_w_mult: 1.5 | Fbeta: 2.0


# **Load Dataset**

In [ ]:
def load_dataset(path: str):
    df = pd.read_csv(path)
    # Check if 'Comment' and 'FinalLabel' columns exist
    if set(df.columns) >= {"Comment","FinalLabel"}:
        df = df[["Comment","FinalLabel"]].rename(columns={"Comment":"text","FinalLabel":"label"})
    # Check if 'text' and 'label' columns exist directly
    elif set(df.columns) >= {"text","label"}:
        df = df[["text","label"]]
    # Fallback to the original 2-column assumption
    elif df.shape[1] == 2:
        df.columns = ["text","label"]
        df = df[df["label"] != "FinalLabel"].copy()
    else:
        raise ValueError(f"Unrecognized CSV format: {df.columns.tolist()}")

    label2id = {"Non-toxic":0, "Toxic":1}
    if not set(df["label"].unique()).issubset(set(label2id.keys())):
        raise ValueError(f"Labels must be {list(label2id.keys())}, found: {df['label'].unique()}")
    df["label_id"] = df["label"].map(label2id)
    return df, label2id

df, label2id = load_dataset(DATA_PATH)
print("Total shape:", df.shape)
print("Total label distribution:\n", df["label"].value_counts(normalize=True).round(3))
df.head()

Total shape: (6000, 3)
Total label distribution:
 label
Non-toxic    0.912
Toxic        0.088
Name: proportion, dtype: float64


,text,label,label_id
0,@user @user I miss you Valbuş💛💙 Come to İstanb...,Non-toxic,0
1,@user We're extremely glad to see y'all landed...,Non-toxic,0
2,@user Lucky ass goal,Non-toxic,0
3,@user Your deeds for the motherland won't go u...,Non-toxic,0
4,@user @user Μπράβο παιδιά!!!,Non-toxic,0


# **Model Setup**

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
config = AutoConfig.from_pretrained(
    MODEL_NAME, num_labels=2, label2id=label2id, id2label={v:k for k,v in label2id.items()}
)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, config=config).to(device)

class CommentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=160):
        self.texts = texts.tolist()
        self.labels = labels.tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        text = str(self.texts[idx]); label = int(self.labels[idx])
        enc = self.tokenizer(text, truncation=True, padding='max_length', max_length=self.max_length, return_tensors='pt')
        return {'input_ids': enc['input_ids'].squeeze(0),
                'attention_mask': enc['attention_mask'].squeeze(0),
                'labels': torch.tensor(label, dtype=torch.long)}


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: /content/drive/MyDrive/Toxic language in football Research/models/Greek-BERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
bert.pooler.dense.bias                     | MISSING    | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 
bert.pooler.dense.weight                   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect ide

# **Train-Test Split**

In [ ]:

# First: train (70%) vs temp (30%)
train_df, temp_df = train_test_split(
    df, test_size=0.30, stratify=df["label_id"], random_state=SEED
)
# Then: temp -> val (15%) and test (15%) i.e., split temp 50/50 stratified
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df["label_id"], random_state=SEED
)

def pct(x): return (x.value_counts(normalize=True).rename({0:"non-toxic",1:"toxic"})*100).round(1)

print("Split sizes:", len(train_df), len(val_df), len(test_df))
print("Train dist (%):\n", pct(train_df["label_id"]))
print("Val   dist (%):\n", pct(val_df["label_id"]))
print("Test  dist (%):\n", pct(test_df["label_id"]))

train_ds = CommentDataset(train_df["text"], train_df["label_id"], tokenizer, MAX_LEN)
val_ds   = CommentDataset(val_df["text"],   val_df["label_id"],   tokenizer, MAX_LEN)
test_ds  = CommentDataset(test_df["text"],  test_df["label_id"],  tokenizer, MAX_LEN)

# Sampler (train only)
if USE_SAMPLER:
    y_train = train_df["label_id"].values
    class_sample_count = np.bincount(y_train)     # e.g., [#non, #tox]
    weights = 1.0 / class_sample_count
    sample_weights = weights[y_train]
    sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SZ, sampler=sampler)
else:
    train_loader = DataLoader(train_ds, batch_size=BATCH_SZ, shuffle=True)

val_loader   = DataLoader(val_ds,   batch_size=BATCH_SZ)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SZ)


Split sizes: 4200 900 900
Train dist (%):
 label_id
non-toxic    91.2
toxic         8.8
Name: proportion, dtype: float64
Val   dist (%):
 label_id
non-toxic    91.2
toxic         8.8
Name: proportion, dtype: float64
Test  dist (%):
 label_id
non-toxic    91.2
toxic         8.8
Name: proportion, dtype: float64


## *Class Weights*

In [ ]:
class_weights = compute_class_weight(class_weight="balanced", classes=np.array([0,1]), y=train_df["label_id"].values)
class_weights = torch.tensor(class_weights, dtype=torch.float)
class_weights[1] = class_weights[1] * TOXIC_WEIGHT_MULT  # boost toxic
class_weights = class_weights.to(device)
print("Class weights (train) with boost:", class_weights.tolist())

Class weights (train) with boost: [0.5483028888702393, 8.513513565063477]


## *Focal Loss*

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, reduction="mean"):
        super().__init__()
        self.alpha = alpha; self.gamma = gamma; self.reduction = reduction
    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, weight=self.alpha, reduction="none")
        pt = torch.exp(-ce)
        loss = (1 - pt) ** self.gamma * ce
        if self.reduction == "mean": return loss.mean()
        if self.reduction == "sum": return loss.sum()
        return loss

if LOSS_TYPE.lower() == "focal":
    loss_fn = FocalLoss(alpha=class_weights, gamma=FOCAL_GAMMA)
else:
    loss_fn = lambda logits, targets: F.cross_entropy(logits, targets, weight=class_weights)
loss_fn

FocalLoss()

# **Evaluation Setup**

In [ ]:
@torch.no_grad()
def evaluate(model, dataloader, device, report=False, threshold=None):
    model.eval()
    all_probs, all_preds, all_true = [], [], []

    running_loss = 0.0
    for batch in dataloader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits  = outputs.logits
        loss    = F.cross_entropy(logits, labels)
        running_loss += loss.item()

        probs = torch.softmax(logits, dim=-1)[:, 1]
        all_probs.extend(probs.detach().cpu().numpy())

        preds = torch.argmax(logits, dim=1) if threshold is None else (probs >= threshold).long()
        all_preds.extend(preds.detach().cpu().numpy())
        all_true.extend(labels.detach().cpu().numpy())

    avg_loss = running_loss / max(1, len(dataloader))

    acc       = accuracy_score(all_true, all_preds)
    f1_macro  = f1_score(all_true, all_preds, average="macro",  zero_division=0)
    f1_toxic  = f1_score(all_true, all_preds, pos_label=1,      zero_division=0)
    prec_tox  = precision_score(all_true, all_preds, pos_label=1, zero_division=0)
    rec_tox   = recall_score(all_true, all_preds, pos_label=1,    zero_division=0)

    if report:
        print(classification_report(all_true, all_preds, digits=2))

    return {
        "loss": avg_loss,
        "acc": acc,
        "f1_macro": f1_macro,
        "f1_toxic": f1_toxic,
        "precision_toxic": prec_tox,
        "recall_toxic": rec_tox,
        "y_true": np.array(all_true),
        "y_probs": np.array(all_probs),
    }

def best_threshold_fbeta(y_true, p_toxic, beta=2.0):
    prec, rec, thr = precision_recall_curve(y_true, p_toxic)
    fbeta = (1+beta**2) * (prec*rec) / (beta**2 * prec + rec + 1e-9)
    if len(thr) == 0: return 0.5
    return float(thr[int(np.nanargmax(fbeta))])


# **Training**

In [ ]:
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
total_steps = len(train_loader) * EPOCHS
num_warmup = int(WARMUP_RATIO * total_steps)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=num_warmup, num_training_steps=total_steps)

best_metric = -1.0
best_state = None
patience_ctr = 0

for epoch in range(1, EPOCHS+1):
    model.train(); running_loss = 0.0
    for batch in train_loader:
        optimizer.zero_grad(set_to_none=True)
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)
        out = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = loss_fn(out.logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step(); scheduler.step()
        running_loss += loss.item()
    train_loss = running_loss / max(1, len(train_loader))

    val_metrics = evaluate(model, val_loader, device, report=False, threshold=None)
    print(
    f"Epoch {epoch:>2} | "
    f"TrainLoss {train_loss:.4f} | "
    f"ValAcc {val_metrics['acc']:.4f} | "
    f"ValF1_macro {val_metrics['f1_macro']:.4f} | "
    f"ValF1_toxic {val_metrics['f1_toxic']:.4f} | "
    f"ValPrec_toxic {val_metrics['precision_toxic']:.4f} | "
    f"ValRec_toxic {val_metrics['recall_toxic']:.4f}"
    )
    try:
      val_auroc = roc_auc_score(val_metrics["y_true"], val_metrics["y_probs"])
      print(f"Val AUROC: {val_auroc:.4f}")
    except ValueError:
      print("Val AUROC: not defined (only one class present in y_true)")

    current = val_metrics["f1_macro"]
    if current > best_metric:
        best_metric = current; patience_ctr = 0
        best_state = {k: v.cpu() for k,v in model.state_dict().items()}
    else:
        patience_ctr += 1
        if patience_ctr > PATIENCE:
            print(f"Early stopping at epoch {epoch}. Best Val macro-F1: {best_metric:.4f}")
            break

if best_state is not None:
    model.load_state_dict(best_state); model.to(device)
    print("Loaded best checkpoint (Val macro-F1=%.4f)" % best_metric)

Epoch  1 | TrainLoss 0.6110 | ValAcc 0.1444 | ValF1_macro 0.1436 | ValF1_toxic 0.1703 | ValPrec_toxic 0.0931 | ValRec_toxic 1.0000
Val AUROC: 0.8198
Epoch  2 | TrainLoss 0.3089 | ValAcc 0.5167 | ValF1_macro 0.4482 | ValF1_toxic 0.2539 | ValPrec_toxic 0.1468 | ValRec_toxic 0.9367
Val AUROC: 0.8358
Epoch  3 | TrainLoss 0.2101 | ValAcc 0.6856 | ValF1_macro 0.5523 | ValF1_toxic 0.3081 | ValPrec_toxic 0.1909 | ValRec_toxic 0.7975
Val AUROC: 0.8148
Epoch  4 | TrainLoss 0.1660 | ValAcc 0.6833 | ValF1_macro 0.5522 | ValF1_toxic 0.3099 | ValPrec_toxic 0.1916 | ValRec_toxic 0.8101
Val AUROC: 0.8061
Epoch  5 | TrainLoss 0.1516 | ValAcc 0.6956 | ValF1_macro 0.5532 | ValF1_toxic 0.3010 | ValPrec_toxic 0.1885 | ValRec_toxic 0.7468
Val AUROC: 0.8147
Epoch  6 | TrainLoss 0.1290 | ValAcc 0.7267 | ValF1_macro 0.5560 | ValF1_toxic 0.2807 | ValPrec_toxic 0.1825 | ValRec_toxic 0.6076
Val AUROC: 0.7962
Epoch  7 | TrainLoss 0.1156 | ValAcc 0.7478 | ValF1_macro 0.5697 | ValF1_toxic 0.2928 | ValPrec_toxic 0.19

# **Validation and Testing**

In [ ]:
# Tune threshold on VALIDATION set (Fbeta)
val_eval  = evaluate(model, val_loader, device, report=False, threshold=None)
best_thr  = best_threshold_fbeta(val_eval["y_true"], val_eval["y_probs"], beta=FBETA)
val_tuned = evaluate(model, val_loader, device, report=True, threshold=best_thr)

print("\n=== Validation (for tuning on Fbeta) ===")
print(f"Default thr=0.5 -> Val Acc: {val_eval['acc']:.4f} | Val Macro-F1: {val_eval['f1_macro']:.4f} | Val F1-toxic: {val_eval['f1_toxic']:.4f}")
print(f"Best thr={best_thr:.3f} (beta={FBETA}) -> Val Acc: {val_tuned['acc']:.4f} | Val Macro-F1: {val_tuned['f1_macro']:.4f} | Val F1-toxic: {val_tuned['f1_toxic']:.4f}")
try:
    val_auroc = roc_auc_score(val_eval["y_true"], val_eval["y_probs"])
    print(f"Val AUROC: {val_auroc:.4f}")
except ValueError:
    print("Val AUROC: not defined (only one class present)")

# FINAL TEST (use the tuned threshold)
test_eval_default = evaluate(model, test_loader, device, report=True, threshold=None)
test_eval_tuned   = evaluate(model, test_loader, device, report=True, threshold=best_thr)

print("\n=== FINAL TEST (unseen) ===")
print(f"Default thr=0.5 -> Test Acc: {test_eval_default['acc']:.4f} | Test Macro-F1: {test_eval_default['f1_macro']:.4f} | Test F1-toxic: {test_eval_default['f1_toxic']:.4f}")
print(f"Tuned thr={best_thr:.3f} (beta={FBETA}) -> Test Acc: {test_eval_tuned['acc']:.4f} | Test Macro-F1: {test_eval_tuned['f1_macro']:.4f} | Test F1-toxic: {test_eval_tuned['f1_toxic']:.4f}")
try:
    test_auroc = roc_auc_score(test_eval_default["y_true"], test_eval_default["y_probs"])
    print(f"Test AUROC: {test_auroc:.4f}")
except ValueError:
    print("Test AUROC: not defined (only one class present)")

              precision    recall  f1-score   support

           0       0.98      0.67      0.80       821
           1       0.20      0.84      0.32        79

    accuracy                           0.69       900
   macro avg       0.59      0.75      0.56       900
weighted avg       0.91      0.69      0.75       900


=== Validation (for tuning on Fbeta) ===
Default thr=0.5 -> Val Acc: 0.7478 | Val Macro-F1: 0.5697 | Val F1-toxic: 0.2928
Best thr=0.063 (beta=2.0) -> Val Acc: 0.6867 | Val Macro-F1: 0.5577 | Val F1-toxic: 0.3188
Val AUROC: 0.8013
              precision    recall  f1-score   support

           0       0.95      0.74      0.83       821
           1       0.18      0.61      0.28        79

    accuracy                           0.72       900
   macro avg       0.57      0.67      0.55       900
weighted avg       0.88      0.72      0.78       900

              precision    recall  f1-score   support

           0       0.97      0.67      0.79       821
     

In [ ]:
# Αποθήκευση του τελικού Fine-Tuned μοντέλου (Classifier)
FINAL_OUTPUT_DIR = "./final_toxic_classifier"

model.save_pretrained(FINAL_OUTPUT_DIR)
tokenizer.save_pretrained(FINAL_OUTPUT_DIR)

print(f"Το τελικό μοντέλο αποθηκεύτηκε στο {FINAL_OUTPUT_DIR}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Το τελικό μοντέλο αποθηκεύτηκε στο ./final_toxic_classifier
